# XL-Sum → mBART50 quality-first training pipeline

This notebook is tuned to be **as close as practical to the XL-Sum paper setup**, while still using **mBART-50** instead of mT5.

## What is matched from the paper
- multilingual summarization on XL-Sum
- `max_source_length = 512`
- `max_target_length = 64`
- beam search with `num_beams = 4`
- `length_penalty = 0.6`
- multilingual low-resource upweighting inspired by the paper's **α = 0.5** smoothing
- fixed validation/test slices of **up to 500 examples per language**

## What is intentionally different
- the paper uses **mT5-base**, not mBART50
- the paper trains on **44 languages**, here you train on your selected subset
- the paper samples **single-language batches**; here we closely approximate that setup with **α = 0.5 resampling**
- we use Hugging Face Trainer for practicality on Colab

This means you can get a **strong reproduction-style pipeline**, but not a guaranteed exact replication of the paper's quality.

In [ ]:
!pip -q install -U   "transformers==4.44.2"   "datasets==2.21.0"   "evaluate==0.4.2"   "sacrebleu==2.4.3"   "sentencepiece==0.2.0"   "accelerate==0.34.2"   "rouge-score==0.1.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the pack

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
from datasets import load_dataset, DatasetDict, concatenate_datasets
import evaluate

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA A100-SXM4-40GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIG
# =========================================================

XLSUM_TO_MBAR = {
    "arabic": "ar_AR",
    "chinese_simplified": "zh_CN",
    "english": "en_XX",
    "french": "fr_XX",
    "hindi": "hi_IN",
    "japanese": "ja_XX",
    "korean": "ko_KR",
    "persian": "fa_IR",
    "russian": "ru_RU",
    "spanish": "es_XX",
    "turkish": "tr_TR",
    "ukrainian": "uk_UA",
    "vietnamese": "vi_VN",
    "portuguese": "pt_XX",
    "indonesian": "id_ID",
}

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
OUTPUT_DIR = "/content/drive/MyDrive/models/mbart_xlsum"

SEED = 42
set_seed(SEED)

# Paper-aligned sequence lengths
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 64

# Paper-aligned decoding
NUM_BEAMS = 4
LENGTH_PENALTY = 0.6
NO_REPEAT_NGRAM_SIZE = 3

# Data split policy
VAL_PER_LANG = 500
TEST_PER_LANG = 500

# Quality-first training
# If your 80GB Colab can handle more, raise PER_DEVICE_TRAIN_BATCH_SIZE.
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch 32

# Optimizer/schedule closer to the paper
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
WARMUP_STEPS = 5000
MAX_STEPS = 35000
LABEL_SMOOTHING = 0.0

# Low-resource language smoothing inspired by the paper
LANG_SMOOTHING_ALPHA = 0.5

# Mixed precision / memory
USE_FP16 = torch.cuda.is_available()
USE_BF16 = False

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Why these defaults?

The XL-Sum paper reports multilingual training with:
- 512-token inputs and 64-token outputs
- beam size 4 and length penalty 0.6
- Adafactor with linear warmup for 5,000 steps and inverse-square-root decay
- multilingual smoothing factor α = 0.5 to upsample lower-resource languages
- 500 dev / 500 test samples per language

Those exact paper choices are reflected below as closely as possible, even though the backbone here is mBART50 rather than mT5.

In [ ]:
# =========================================================
# LOAD SELECTED XL-SUM LANGUAGES
# =========================================================

def add_lang_columns(example, xlsum_lang, mbart_lang):
    example["xlsum_lang"] = xlsum_lang
    example["mbart_lang"] = mbart_lang
    return example

train_parts, val_parts, test_parts = [], [], []

for xlsum_lang, mbart_lang in XLSUM_TO_MBAR.items():
    ds = load_dataset("csebuetnlp/xlsum", xlsum_lang, cache_dir="/content/hf_cache")

    ds["train"] = ds["train"].map(
        lambda ex, xl=xlsum_lang, ml=mbart_lang: add_lang_columns(ex, xl, ml)
    )
    ds["validation"] = ds["validation"].map(
        lambda ex, xl=xlsum_lang, ml=mbart_lang: add_lang_columns(ex, xl, ml)
    )
    ds["test"] = ds["test"].map(
        lambda ex, xl=xlsum_lang, ml=mbart_lang: add_lang_columns(ex, xl, ml)
    )

    val_n = min(VAL_PER_LANG, len(ds["validation"]))
    test_n = min(TEST_PER_LANG, len(ds["test"]))

    train_parts.append(ds["train"])
    val_parts.append(ds["validation"].shuffle(seed=SEED).select(range(val_n)))
    test_parts.append(ds["test"].shuffle(seed=SEED).select(range(test_n)))

raw_dataset = DatasetDict({
    "train": concatenate_datasets(train_parts).shuffle(seed=SEED),
    "validation": concatenate_datasets(val_parts).shuffle(seed=SEED),
    "test": concatenate_datasets(test_parts).shuffle(seed=SEED),
})

raw_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/37519 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4689 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4689 [00:00<?, ? examples/s]

Map:   0%|          | 0/37519 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4780 [00:00<?, ? examples/s]

Map:   0%|          | 0/38242 [00:00<?, ? examples/s]

Map:   0%|          | 0/4780 [00:00<?, ? examples/s]

Map:   0%|          | 0/4780 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 818134
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 7500
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 7500
    })
})

In [ ]:
# Inspect language distribution
train_lang_counts = Counter(raw_dataset["train"]["xlsum_lang"])
val_lang_counts = Counter(raw_dataset["validation"]["xlsum_lang"])
test_lang_counts = Counter(raw_dataset["test"]["xlsum_lang"])

display(pd.DataFrame({
    "language": list(XLSUM_TO_MBAR.keys()),
    "train_count": [train_lang_counts[k] for k in XLSUM_TO_MBAR.keys()],
    "val_count": [val_lang_counts[k] for k in XLSUM_TO_MBAR.keys()],
    "test_count": [test_lang_counts[k] for k in XLSUM_TO_MBAR.keys()],
}).sort_values("train_count", ascending=False))

,language,train_count,val_count,test_count
2,english,306522,500,500
4,hindi,70778,500,500
8,russian,62243,500,500
13,portuguese,57402,500,500
7,persian,47251,500,500
11,ukrainian,43201,500,500
14,indonesian,38242,500,500
9,spanish,38110,500,500
0,arabic,37519,500,500
1,chinese_simplified,37362,500,500


In [ ]:
# =========================================================
# ALPHA=0.5 RESAMPLING (paper-inspired low-resource smoothing)
# =========================================================
# Paper idea:
# p(lang) ∝ n_lang^alpha, with alpha = 0.5
#
# We approximate that by resampling each language's train subset
# to a target size proportional to n_lang^0.5.

def make_alpha_resampled_train(ds, alpha=0.5, total_target_size=None, seed=42):
    lang_counts = Counter(ds["xlsum_lang"])
    langs = list(lang_counts.keys())

    counts = np.array([lang_counts[lang] for lang in langs], dtype=np.float64)
    probs = counts ** alpha
    probs = probs / probs.sum()

    if total_target_size is None:
        total_target_size = len(ds)

    target_counts = np.floor(probs * total_target_size).astype(int)

    # distribute remaining samples
    remainder = total_target_size - target_counts.sum()
    if remainder > 0:
        order = np.argsort(-(probs * total_target_size - target_counts))
        for idx in order[:remainder]:
            target_counts[idx] += 1

    resampled_parts = []
    rng = np.random.default_rng(seed)

    for lang, target_n in zip(langs, target_counts):
        lang_ds = ds.filter(lambda ex, lang=lang: ex["xlsum_lang"] == lang)
        current_n = len(lang_ds)

        if target_n <= current_n:
            part = lang_ds.shuffle(seed=seed).select(range(target_n))
        else:
            # oversample with replacement
            indices = rng.integers(0, current_n, size=target_n).tolist()
            part = lang_ds.select(indices)

        resampled_parts.append(part)

    out = concatenate_datasets(resampled_parts).shuffle(seed=seed)
    return out, pd.DataFrame({
        "language": langs,
        "orig_count": counts.astype(int),
        "target_count": target_counts,
        "orig_prob": counts / counts.sum(),
        "smoothed_prob": probs,
    }).sort_values("orig_count", ascending=False)

resampled_train, resampling_table = make_alpha_resampled_train(
    raw_dataset["train"],
    alpha=LANG_SMOOTHING_ALPHA,
    total_target_size=len(raw_dataset["train"]),
    seed=SEED,
)

display(resampling_table)

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

,language,orig_count,target_count,orig_prob,smoothed_prob
4,english,306522,146125,0.374660,0.178608
1,hindi,70778,70217,0.086512,0.085826
5,russian,62243,65848,0.076079,0.080485
3,portuguese,57402,63235,0.070162,0.077292
0,persian,47251,57372,0.057755,0.070125
11,ukrainian,43201,54858,0.052804,0.067053
10,indonesian,38242,51614,0.046743,0.063087
2,spanish,38110,51525,0.046582,0.062978
8,arabic,37519,51123,0.045859,0.062488
6,chinese_simplified,37362,51016,0.045667,0.062357


In [ ]:
dataset = DatasetDict({
    "train": resampled_train,
    "validation": raw_dataset["validation"],
    "test": raw_dataset["test"],
})

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 818134
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 7500
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 7500
    })
})

In [ ]:
# =========================================================
# TOKENIZER + MODEL
# =========================================================

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

model.gradient_checkpointing_enable()

if torch.cuda.is_available():
    model = model.cuda()

print("Loaded:", MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Loaded: facebook/mbart-large-50-many-to-many-mmt


In [ ]:
# =========================================================
# CLEANING
# =========================================================

def clean_example(example):
    return (
        example["text"] is not None
        and example["summary"] is not None
        and isinstance(example["text"], str)
        and isinstance(example["summary"], str)
        and len(example["text"].strip()) > 0
        and len(example["summary"].strip()) > 0
        and example["mbart_lang"] is not None
    )

dataset = dataset.filter(clean_example)
dataset

Filter:   0%|          | 0/818134 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 818123
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 7500
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text', 'xlsum_lang', 'mbart_lang'],
        num_rows: 7500
    })
})

In [ ]:
# =========================================================
# PREPROCESSING (fixed, fast, mBART50-safe)
# =========================================================

from collections import defaultdict

def preprocess_function(batch):
    grouped = defaultdict(list)

    # aynı dili kullanan örnekleri birlikte topla
    for text, summary, lang_code in zip(batch["text"], batch["summary"], batch["mbart_lang"]):
        grouped[lang_code].append((text, summary))

    final_input_ids = []
    final_attention_masks = []
    final_labels = []

    for lang_code, pairs in grouped.items():
        texts = [x[0] for x in pairs]
        summaries = [x[1] for x in pairs]

        tokenizer.src_lang = lang_code
        tokenizer.tgt_lang = lang_code

        model_inputs = tokenizer(
            texts,
            text_target=summaries,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=False,
        )

        # labels için ayrı target max length uygula
        label_features = tokenizer(
            text_target=summaries,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding=False,
        )

        labels = label_features["input_ids"]

        for inp_ids, attn, lab in zip(
            model_inputs["input_ids"],
            model_inputs["attention_mask"],
            labels,
        ):
            if inp_ids is None or attn is None or lab is None:
                continue
            if any(x is None for x in inp_ids):
                continue
            if any(x is None for x in attn):
                continue
            if any(x is None for x in lab):
                continue

            final_input_ids.append(inp_ids)
            final_attention_masks.append(attn)
            final_labels.append(lab)

    return {
        "input_ids": final_input_ids,
        "attention_mask": final_attention_masks,
        "labels": final_labels,
    }

tokenized = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=256,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)

tokenized

Tokenizing:   0%|          | 0/818123 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/7500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818123
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7500
    })
})

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

In [ ]:
# =========================================================
# ROUGE
# =========================================================

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,
    )
    result = {k: round(v * 100, 4) for k, v in result.items()}

    pred_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = round(float(np.mean(pred_lens)), 2)
    return result

In [ ]:
# =========================================================
# TRAINING ARGS
# =========================================================

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,

    do_train=True,
    do_eval=True,

    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    label_smoothing_factor=LABEL_SMOOTHING,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    optim="adafactor",
    lr_scheduler_type="inverse_sqrt",

    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",

    eval_steps=5000,
    save_steps=5000,
    logging_steps=100,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=NUM_BEAMS,

    fp16=USE_FP16,
    bf16=USE_BF16,

    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


In [ ]:
train_result = trainer.train()
train_result

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
5000,2.697800,2.642181,19.225100,6.446200,15.330500,15.319100,34.260000
10000,2.343900,2.413790,21.034100,7.517800,16.957600,16.964400,33.710000
15000,2.200000,2.316701,21.929100,8.059500,17.807200,17.825100,34.310000
20000,2.072700,2.269275,22.406800,8.409600,18.141300,18.139700,36.110000
25000,1.958800,2.263595,22.688600,8.508600,18.449200,18.448300,36.410000
30000,1.666400,2.298910,22.892500,8.623400,18.600600,18.618800,36.250000
35000,1.611300,2.311783,22.849400,8.739400,18.507700,18.509100,36.410000


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#sav

TrainOutput(global_step=35000, training_loss=2.1508960488455635, metrics={'train_runtime': 43614.9665, 'train_samples_per_second': 25.679, 'train_steps_per_second': 0.802, 'total_flos': 1.2134343341254902e+18, 'train_loss': 2.1508960488455635, 'epoch': 1.3689789372812078})

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


Saved to: /content/drive/MyDrive/models/mbart_xlsum


In [ ]:
val_metrics = trainer.evaluate(tokenized["validation"], metric_key_prefix="val")
val_metrics

{'val_loss': 2.298909902572632,
 'val_rouge1': 22.8925,
 'val_rouge2': 8.6234,
 'val_rougeL': 18.6006,
 'val_rougeLsum': 18.6188,
 'val_gen_len': 36.25,
 'val_runtime': 1511.1099,
 'val_samples_per_second': 4.963,
 'val_steps_per_second': 0.621,
 'epoch': 1.3689789372812078}

In [ ]:
test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
test_metrics

{'test_loss': 2.3320047855377197,
 'test_rouge1': 22.8033,
 'test_rouge2': 8.6503,
 'test_rougeL': 18.5107,
 'test_rougeLsum': 18.5316,
 'test_gen_len': 36.23,
 'test_runtime': 1498.2176,
 'test_samples_per_second': 5.006,
 'test_steps_per_second': 0.626,
 'epoch': 1.3689789372812078}

In [ ]:
# =========================================================
# PER-LANGUAGE TEST EVALUATION
# =========================================================

per_lang_rows = []

for xlsum_lang, mbart_lang in XLSUM_TO_MBAR.items():
    lang_raw = dataset["test"].filter(lambda ex, lang=xlsum_lang: ex["xlsum_lang"] == lang)
    lang_tok = lang_raw.map(
        preprocess_function,
        batched=True,
        remove_columns=lang_raw.column_names,
        desc=f"Tokenizing {xlsum_lang} test set"
    )
    metrics = trainer.evaluate(lang_tok, metric_key_prefix=f"test_{xlsum_lang}")
    metrics["language"] = xlsum_lang
    per_lang_rows.append(metrics)

per_lang_df = pd.DataFrame(per_lang_rows)
display(per_lang_df)

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing arabic test set:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing chinese_simplified test set:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing portuguese test set:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing indonesian test set:   0%|          | 0/500 [00:00<?, ? examples/s]

,test_arabic_loss,test_arabic_rouge1,test_arabic_rouge2,test_arabic_rougeL,test_arabic_rougeLsum,test_arabic_gen_len,test_arabic_runtime,test_arabic_samples_per_second,test_arabic_steps_per_second,epoch,...,test_portuguese_steps_per_second,test_indonesian_loss,test_indonesian_rouge1,test_indonesian_rouge2,test_indonesian_rougeL,test_indonesian_rougeLsum,test_indonesian_gen_len,test_indonesian_runtime,test_indonesian_samples_per_second,test_indonesian_steps_per_second
0,2.444503,2.15,0.0,2.1167,2.1667,36.23,98.7254,5.065,0.638,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.368979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# =========================================================
# GENERATION HELPER
# =========================================================

@torch.no_grad()
def summarize(text, lang_code="tr_TR", max_new_tokens=64, num_beams=4):
    tokenizer.src_lang = lang_code
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    )
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[lang_code],
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        length_penalty=LENGTH_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

In [ ]:
# Quick sanity check
for idx in [0, 1, 2]:
    ex = dataset["test"][idx]
    pred = summarize(ex["text"], lang_code=ex["mbart_lang"])

    print("=" * 120)
    print("LANG:", ex["xlsum_lang"], ex["mbart_lang"])
    print("REFERENCE:", ex["summary"][:500])
    print("PREDICTION:", pred[:500])
    print()